In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import json
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer

In [ ]:
# Test GPU is available
import torch
print(torch.cuda.is_available()) # Should be true
print(torch.cuda.get_device_name(0)) # If available

In [ ]:
DATA_PATH = "./data/trn.json"
OUTPUT_PATH_DATASET = "./data/formatted_data.json"

max_seq_length = 2048
dtype = None
load_in_4bit = True
fourbit_models = [
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
]

In [ ]:
def format_dataset_into_model_input(data):
    instructions = []
    inputs = []
    outputs = []

    for obj in data:
        instructions.append("DESCRIBE ABOUT THE PRODUCT.")
        inputs.append(obj.get("title", ""))
        outputs.append(obj.get("content", ""))
    
    final_output = {
        "instruction": instructions,
        "input": inputs,
        "output": outputs
    }

    with open(OUTPUT_PATH_DATASET, 'w') as output_file:
        json.dump(final_output, output_file, indent=4)
    
    print(f"Dataset saved in {OUTPUT_PATH_DATASET}")


In [ ]:
# Open file without formatation
with open(DATA_PATH, "r") as f:
    data = [json.loads(line) for line in f]

format_dataset_into_model_input(data)

In [ ]:
# Loading model in GPU memory
model, processor = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True
)

In [ ]:
model.gradient_checkpointing_enable()

In [ ]:
# Get tokenizer from processor
tokenizer = processor

In [ ]:
# Check tokenizer
print(type(processor))

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407
)

In [30]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

In [ ]:
# Formats the dataset into a specific prompt strucutre for fine-tunning a language model.
EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []

    for instruction, input, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text": texts }
pass

dataset = load_dataset("json", data_files = OUTPUT_PATH_DATASET, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
import os
os.environ["TORCHINDUCTOR_DISABLE"] = "1"

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs"
    )
)

In [ ]:
# Clean cache from cuda
import torch
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
# Train the model with dataset
trainer_stats = trainer.train()

In [37]:
inputs = tokenizer(
    [
        alpaca_prompt.format(
            "DESCRIBE ABOUT THE PRODUCT.",
            "The Book of Revelation",
            "",
        )
    ], return_tensors = "pt").to("cuda")

text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<bos>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
DESCRIBE ABOUT THE PRODUCT.

### Input:
The Book of Revelation

### Response:
The book of Revelation tells the story of the heavenly or terrifying conflict between God and all humanity in the last days. The book presents a detailed, often poetic, and deeply symbolic account of the end times, the tribulations of the present time, and the ultimate triumph of good over evil and God's own redemptive power over a host of oppressors, both human and demonic. Its use of imagery is both extremely vivid and profound, but it is a deeply theological book. The authors who composed the book of Revelation are unknown and he is recorded as a monk from Macedonia, who was an elder of the ancient Christian community of Ephesus, the


In [43]:
# Any prompt only to test
prompt = "Tell me only one joke about computers."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens = 128)
print(tokenizer.decode(outputs[0]))

<bos>Tell me only one joke about computers. I don't want a "brain teaser" or a philosophical response.

What do you get when you mix a computer and a banana?
<end_of_turn>
